# Data Downloading

In [ ]:
from pathlib import Path

try:
    REPO_ROOT = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    REPO_ROOT = Path.cwd().resolve()

DATA_DIR = REPO_ROOT / "geom-qm9"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"REPO_ROOT = {REPO_ROOT}")
print(f"DATA_DIR  = {DATA_DIR}")

In [ ]:
!wget -qO- --show-progress https://dataverse.harvard.edu/api/access/datafile/4327252 | tar -xf - -C "{DATA_DIR}"

-                     1%[                    ] 626.30M  41.4MB/s    eta 19m 4s ^C


In [ ]:
import json
import os
import pickle
import random

import numpy as np
import torch


def rotate_coordinates(coords):
    alpha, beta, gamma = np.random.uniform(0, 2 * np.pi, size=3)
    R_x = np.array([[1, 0, 0],
                    [0, np.cos(alpha), -np.sin(alpha)],
                    [0, np.sin(alpha), np.cos(alpha)]])

    R_y = np.array([[np.cos(beta), 0, np.sin(beta)],
                    [0, 1, 0],
                    [-np.sin(beta), 0, np.cos(beta)]])

    R_z = np.array([[np.cos(gamma), -np.sin(gamma), 0],
                    [np.sin(gamma), np.cos(gamma), 0],
                    [0, 0, 1]])

    R = R_z @ R_y @ R_x

    return coords @ R.T

In [ ]:
def seed_everything(seed_value=42):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    os.environ['PYTHONHASHSEED'] = str(seed_value)
    os.environ['PL_GLOBAL_SEED'] = str(seed_value)
    os.environ['PL_SEED_WORKERS'] = '1'

    print(f"Global random seed set to {seed_value}")

seed_everything(42)

Global random seed set to 42


In [ ]:
base_path = str(DATA_DIR)
qm9_file = os.path.join(base_path, "rdkit_folder/summary_qm9.json")

with open(qm9_file, "r") as f:
    qm9_summ = json.load(f)

In [ ]:
random_smiles = list(qm9_summ.keys())
random.shuffle(random_smiles)
len(random_smiles)

133258

# Data Preparation

In [ ]:
smiles_30 = []

for smiles in random_smiles:
    if qm9_summ[smiles].get("uniqueconfs", 0) >= 30:
        smiles_30.append(smiles)
    if len(smiles_30) == 30:
        break

smiles_30

['O[C@H]1C[C@H]1CCC1CC1',
 'C[C@H](C=O)CO[CH][NH]',
 'CC[C@@H](O)[C@@H](C)CCO',
 'CC[C@H]1CC[C@@](C)(O)C1',
 'C[C@H]1CCO[C@H]1CCO',
 'OCC[C@H]1C[C@H]2OC[C@@H]12',
 'CC(=O)[C@@H](C)CC1CC1',
 'CCC[C@@H](CO)N1CC1',
 'C#CC[C@](C)(O)CC=O',
 'O=CC[C@@H]1CO[C@@H]1CO',
 'O=CNC[C@H](O)[C@H]1CO1',
 'O=CCCC#C[C@H]1CN1',
 'C[C@@H](CO)CCNC=O',
 'CC#CCOC[C@@H](C)O',
 'CCN[C@@H](C#N)C(N)=O',
 'CCC[C@@H](C)COC=O',
 'CCc1ccc(CO)[nH]1',
 'CC(=O)C[C@@]1(O)C[C@H]1O',
 'CCOC(=O)[C@@H](C)N',
 'CCCCCNC=O',
 'C[C@@H]1C=CC[C@H](CO)C1',
 'CC(C)[C@@H](O)C1(O)CC1',
 'N#CCOCCC1CC1',
 'C[C@H]1N[C@]1(CO)CC=O',
 'OCC1=CCCCOC1',
 'C[C]([NH])OCCCCO',
 'CC(=O)[C@H](C=O)[C@@H](C)O',
 'OCC[C@H]1C[C@@H](O)C1',
 'C[C@@H](CO)[C@@H](O)[C@H]1CN1',
 'CC[C@@H](COC)N1CC1']

In [ ]:
def extract_top_num_conformers(smiles_list, num=30):
    results = {}
    for smiles in smiles_list:
        meta = qm9_summ.get(smiles)
        pickle_file = os.path.join(base_path, "rdkit_folder", meta['pickle_path'])
        with open(pickle_file, 'rb') as f:
            mol_data = pickle.load(f)

        confs_list = sorted(mol_data.get('conformers', []), key=lambda x: x.get('relativeenergy', 0))
        conformers_info = []

        for conf_entry in confs_list[:num]:
            rd_mol = conf_entry.get('rd_mol')
            energy = conf_entry.get('relativeenergy')

            if rd_mol:
                atom_nums = [atom.GetAtomicNum() for atom in rd_mol.GetAtoms()]
                rd_conf = rd_mol.GetConformer()
                positions = rotate_coordinates(rd_conf.GetPositions())
                xyz = np.column_stack((atom_nums, positions))

                conformers_info.append({'energy': energy, 'xyz': xyz})
        results[smiles] = conformers_info
    return results

smiles_30_confs = extract_top_num_conformers(smiles_30, num=30)

In [ ]:
def save_to_xyz_xmol(molecule_data_dict, base_output_dir):
    periodic_table = {1: 'H', 6: 'C', 7: 'N', 8: 'O', 9: 'F', 16: 'S', 17: 'Cl'}

    for smiles, conformers in molecule_data_dict.items():
        safe_name = "".join([c if c.isalnum() else "_" for c in smiles])
        molecule_dir = os.path.join(base_output_dir, safe_name)
        os.makedirs(molecule_dir, exist_ok=True)

        for i, conf in enumerate(conformers):
            xyz_data = conf['xyz']
            num_atoms = len(xyz_data)
            filename = os.path.join(molecule_dir, f"{safe_name}_conf_{i}.xyz")

            with open(filename, 'w') as f:
                f.write(f"{num_atoms}\n")
                f.write(f"Energy: {conf['energy']}\n")

                for row in xyz_data:
                    atom_num = int(row[0])
                    symbol = periodic_table.get(atom_num, str(atom_num))
                    x, y, z = row[1], row[2], row[3]
                    f.write(f"{symbol:2} {x:12.8f} {y:12.8f} {z:12.8f}\n")

save_to_xyz_xmol(smiles_30_confs, str(DATA_DIR / "output_30_confs")) 